# Customer Churn — Logistic Regression

Exploratory analysis, training, evaluation, and model export for the included Telco Customer Churn dataset. See [README.md](README.md) for setup, baseline results, and limitations.


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

%matplotlib inline


In [ ]:
# Load the included dataset from the project working directory
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.shape


In [ ]:
# Quick look
display(df.head())
print('Info:')
print(df.info())
print('Missing values by column:')
print(df.isnull().sum())
print('Target distribution:')
print(df['Churn'].value_counts())
print('Target distribution (pct):')
print(df['Churn'].value_counts(normalize=True))


In [ ]:
# Basic cleaning: convert TotalCharges to numeric (some entries may be spaces) and drop rows with missing TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_tc = df['TotalCharges'].isnull().sum()
print('TotalCharges missing:', missing_tc)
if missing_tc > 0:
    df = df[df['TotalCharges'].notnull()].copy()

# Drop customerID (identifier)
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Map target to binary
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

# Basic numeric cast for TotalCharges and MonthlyCharges and tenure
for c in ['tenure','MonthlyCharges','TotalCharges']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df.shape


In [ ]:
# One-hot encode categorical features (drop_first to avoid collinearity)
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print('Categorical cols to encode:', cat_cols)
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print('Encoded shape:', df_encoded.shape)

# Separate X and y
y = df_encoded['Churn']
X = df_encoded.drop(columns=['Churn'])

# Train-test split (stratify on target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


In [ ]:
# Train logistic regression
model = LogisticRegression(max_iter=1000, solver='lbfgs')
model.fit(X_train, y_train)

# Predictions and probabilities
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

# Evaluation metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print('Accuracy: {:.4f}'.format(acc))
print('Precision: {:.4f}'.format(prec))
print('Recall: {:.4f}'.format(rec))
print('F1-score: {:.4f}'.format(f1))
print('ROC AUC: {:.4f}'.format(roc_auc))

print('Classification report:')
print(classification_report(y_test, y_pred))

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))


In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label='ROC curve (area = %0.3f)' % roc_auc)
plt.plot([0,1],[0,1],'k--')
plt.xlim([0.0,1.0])
plt.ylim([0.0,1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


## Notes and next steps
- This notebook performs a single baseline logistic regression.
- For production-ready models consider: feature scaling, cross-validation, hyperparameter tuning, handling class imbalance (e.g., class weights or resampling), and more careful feature engineering.
- For reproducibility, ensure the same preprocessing steps are applied when scoring new data (the original churn_analysis.ipynb has more EDA and alternative preprocessing that may be useful).


In [ ]:
# --- Option A: dict-style input (user-friendly) ---
# Provide values using the original feature names (as in the raw dataset)
test_raw = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 12,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 70.70,
    'TotalCharges': 845.50
}

# Build DataFrame for the single row
test_df = pd.DataFrame([test_raw])

# Ensure numeric conversions for numeric columns
for num_col in ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']:
    if num_col in test_df.columns:
        test_df[num_col] = pd.to_numeric(test_df[num_col], errors='coerce')

# Encode the observed categories before aligning to the training columns
# Keep drop_first=False for a single customer so categorical inputs are retained.
test_encoded = pd.get_dummies(test_df, columns=cat_cols, drop_first=False)

# Align columns to training features (X.columns) and fill missing columns with 0
test_encoded = test_encoded.reindex(columns=X.columns, fill_value=0)

# Predict
proba = model.predict_proba(test_encoded)[:, 1][0]
label = model.predict(test_encoded)[0]

print(f'Dict input -> Predicted probability of churn: {proba:.4f}')
print(f'Dict input -> Predicted label (1=Churn): {label}')

### Feature Analysis

In [ ]:
# Number of top features to display in the bar plot
top_n = 20

# Ensure model has coef_ (binary logistic regression)
if not hasattr(model, 'coef_'):
    raise AttributeError("The trained model does not expose 'coef_'. Ensure it's a sklearn LogisticRegression instance.")

# Extract coefficients (shape: (1, n_features) for binary)
coefs = model.coef_.ravel()
features = X.columns.to_list()

# Build DataFrame with coefficient info and simple feature-statistics for interpretation
feat_df = pd.DataFrame({
    'feature': features,
    'coef': coefs,
})
feat_df['abs_coef'] = feat_df['coef'].abs()
feat_df['odds_ratio'] = np.exp(feat_df['coef'])  # how odds change per unit increase
# use mean for numeric columns, and for binary/one-hot columns mean ~ prevalence
feat_df['mean_value'] = X.mean().values

# Sort by absolute importance
feat_df = feat_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)

# Print top positive and top negative drivers
top_pos = feat_df[feat_df['coef'] > 0].head(10)
top_neg = feat_df[feat_df['coef'] < 0].head(10)

print("Top positive coefficients (increase predicted churn):")
print(top_pos[['feature','coef','odds_ratio','mean_value']].head(10).to_string(index=False))
print("\nTop negative coefficients (decrease predicted churn):")
print(top_neg[['feature','coef','odds_ratio','mean_value']].head(10).to_string(index=False))

# Bar plot of top_n features by absolute coefficient (signed bars)
plot_df = feat_df.head(top_n).sort_values('coef')  # sort by coef so negatives on left, positives on right
plt.figure(figsize=(10, max(6, top_n * 0.35)))
colors = ['#2ca02c' if v < 0 else '#d62728' for v in plot_df['coef']]  # green for negative (protective), red for positive (risk)
plt.barh(plot_df['feature'], plot_df['coef'], color=colors)
plt.axvline(0, color='k', linewidth=0.8)
plt.xlabel('Log-odds coefficient (positive => increases churn probability)')
plt.title(f'Top {top_n} features by absolute logistic regression coefficient')
plt.grid(axis='x', linestyle='--', alpha=0.4)

# Annotate bars with odds ratio (rounded)
for i, (coef, oratio) in enumerate(zip(plot_df['coef'], plot_df['odds_ratio'])):
    plt.text(coef + (0.02 if coef >= 0 else -0.02), i, f'OR={oratio:.2f}', va='center',
             ha=('left' if coef >= 0 else 'right'), color='black', fontsize=8)

plt.tight_layout()
plt.show()

# Optional: show full table of features sorted by importance
feat_df_display = feat_df.copy()
feat_df_display['coef'] = feat_df_display['coef'].round(4)
feat_df_display['odds_ratio'] = feat_df_display['odds_ratio'].round(3)
feat_df_display['mean_value'] = feat_df_display['mean_value'].round(3)
display(feat_df_display.head(50))  # show top 50 for quick inspection

### Saving the model

In [ ]:
from joblib import dump, load
from pathlib import Path

# Save a single artifact containing model + metadata
out_path = Path.cwd() / "churn_logistic_model.pkl"
artifact = {
    "model": model,
    "feature_names": X.columns.to_list(),
    "cat_cols": cat_cols
}

dump(artifact, out_path)
print(f"Saved model artifact to: {out_path}")



In [ ]:
# Load saved model artifact and predict from a user-provided dict (test_raw)
from joblib import load
import pandas as pd
import numpy as np
import os

# Path to the saved artifact (use the same path you used to save)
pkl_path = out_path

# Load artifact
artifact = load(pkl_path)
model = artifact["model"]
feature_names = artifact["feature_names"]  # list of columns the model expects (X.columns)
cat_cols = artifact["cat_cols"]            # original categorical column names used for get_dummies

# Example user-provided dict (replace values with your test case)
test_raw = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 12,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 70.70,
    'TotalCharges': 845.50
}

# Build DataFrame for the single row
test_df = pd.DataFrame([test_raw])

# Coerce numeric columns that might be strings
for num_col in ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']:
    if num_col in test_df.columns:
        test_df[num_col] = pd.to_numeric(test_df[num_col], errors='coerce')

# Encode all observed categories; reindexing removes reference categories
test_encoded = pd.get_dummies(test_df, columns=cat_cols, drop_first=False)

# Align to the model's feature ordering; missing cols filled with 0
test_encoded = test_encoded.reindex(columns=feature_names, fill_value=0)

# Optional: check for any NaNs produced by coercion
if test_encoded.isnull().any().any():
    print("Warning: NaNs present in test row after processing. Check numeric fields.")
    print(test_encoded[test_encoded.isnull().any(axis=1)])

# Predict probability and class
proba = model.predict_proba(test_encoded)[:, 1][0]
label = model.predict(test_encoded)[0]

print(f"Predicted probability of churn: {proba:.4f}")
print(f"Predicted label (1=Churn): {label}")

In [ ]:
# Load saved model artifact and predict from a user-provided dict (test_raw)
from joblib import load
import pandas as pd
import numpy as np
import os

# Path to the saved artifact (use the same path you used to save)
pkl_path = out_path

# Load artifact
artifact = load(pkl_path)
model = artifact["model"]
feature_names = artifact["feature_names"]  # list of columns the model expects (X.columns)
cat_cols = artifact["cat_cols"]            # original categorical column names used for get_dummies

# Example user-provided dict (replace values with your test case)
test_raw = {
    'gender': 'Male',
    'SeniorCitizen': 1,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 1,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 99.70,
    'TotalCharges': 99.50
}

# Build DataFrame for the single row
test_df = pd.DataFrame([test_raw])

# Coerce numeric columns that might be strings
for num_col in ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']:
    if num_col in test_df.columns:
        test_df[num_col] = pd.to_numeric(test_df[num_col], errors='coerce')

# Encode all observed categories; reindexing removes reference categories
test_encoded = pd.get_dummies(test_df, columns=cat_cols, drop_first=False)

# Align to the model's feature ordering; missing cols filled with 0
test_encoded = test_encoded.reindex(columns=feature_names, fill_value=0)

# Optional: check for any NaNs produced by coercion
if test_encoded.isnull().any().any():
    print("Warning: NaNs present in test row after processing. Check numeric fields.")
    print(test_encoded[test_encoded.isnull().any(axis=1)])

# Predict probability and class
proba = model.predict_proba(test_encoded)[:, 1][0]
label = model.predict(test_encoded)[0]

print(f"Predicted probability of churn: {proba:.4f}")
print(f"Predicted label (1=Churn): {label}")